In [1]:
"""
ERCOT RTM LMP 电价预测 - 完整特征工程模块 (含DAM特征)
======================================================
数据源: 
  - RTM: ERCOT 2015-2024年 15分钟频率
  - DAM: ERCOT 2015-2024年 1小时频率

预测目标: 未来15/30/45/60分钟RTM电价 (horizon=1/2/3/4)
特征数量: 68个RTM特征 + 12个DAM特征 = 80个特征

关键点:
  - 预测时刻t的未来RTM时，对应的DAM价格已知（前一天出清）
  - DAM特征包括：当前DAM、目标时刻DAM、RTM-DAM价差等

作者: 李源
版本: 4.0
"""

# =============================================================================
# RTM特征 (68个)
# =============================================================================

# --- 时间特征 (7个) ---
# 0  hour                    [CAT] 小时 (0-23)
# 1  minute                  [CAT] 分钟 (0, 15, 30, 45)
# 2  day_of_week             [CAT] 星期几 (0-6)
# 3  day_of_month            [CAT] 日期 (1-31)
# 4  month                   [CAT] 月份 (1-12)
# 5  quarter                 [CAT] 季度 (1-4)
# 6  week_of_year                  年内第几周

# --- 时间标记 (7个) ---
# 7  is_weekend              [CAT] 是否周末
# 8  is_peak_hour            [CAT] 是否峰时 7:00-22:00
# 9  is_summer               [CAT] 是否夏季 6-9月
# 10 is_super_peak           [CAT] 是否超级峰时
# 11 is_holiday              [CAT] 是否节假日
# 12 is_business_day         [CAT] 是否工作日
# 13 hour_block              [CAT] 时段块 (0-3)

# --- 时间交互 (3个) ---
# 14 hour_dow_interaction    [CAT] 小时×星期交互
# 15 is_morning_ramp         [CAT] 是否早高峰爬坡
# 16 is_evening_ramp         [CAT] 是否晚高峰爬坡

# --- 滞后特征 (6个) ---
# 17 price_lag_1                   滞后1步 (15分钟前)
# 18 price_lag_2                   滞后2步 (30分钟前)
# 19 price_lag_3                   滞后3步 (45分钟前)
# 20 price_lag_4                   滞后4步 (1小时前)
# 21 price_lag_96                  滞后96步 (昨日同时刻)
# 22 price_lag_672                 滞后672步 (上周同时刻)

# --- 滞后交互 (2个) ---
# 23 price_lag_4_diff_lag_96       1小时前vs昨日差值
# 24 price_lag_672_diff_lag_96     上周vs昨日差值

# --- 滚动统计 (13个) ---
# 25 price_mean_1h                 1小时均价
# 26 price_std_1h                  1小时标准差
# 27 price_max_1h                  1小时最高价
# 28 price_min_1h                  1小时最低价
# 29 price_mean_3h                 3小时均价
# 30 price_std_3h                  3小时标准差
# 31 price_mean_24h                24小时均价
# 32 price_std_24h                 24小时标准差
# 33 price_max_24h                 24小时最高价
# 34 price_min_24h                 24小时最低价
# 35 price_skew_24h                24小时偏度
# 36 price_mean_7d                 7天均价
# 37 price_std_7d                  7天标准差

# --- 同时刻统计 (2个) ---
# 38 same_time_mean                过去7天同时刻均价
# 39 same_time_dow_mean            过去4周同星期同时刻均价

# --- 动量特征 (8个) ---
# 40 price_diff_1                  1步差分
# 41 price_diff_4                  4步差分
# 42 price_diff_96                 96步差分
# 43 price_pct_1                   1步变化率
# 44 price_momentum_1h             1小时动量 (Z-score)
# 45 price_momentum_24h            24小时动量 (Z-score)
# 46 price_vs_mean_24h             当前/24小时均价
# 47 price_acceleration            加速度

# --- 尖峰特征 (13个) ---
# 48 is_spike_100            [CAT] 是否>$100
# 49 spike_count_24h_100           24小时>$100次数
# 50 spike_count_7d_100            7天>$100次数
# 51 is_spike_300            [CAT] 是否>$300
# 52 spike_count_24h_300           24小时>$300次数
# 53 spike_count_7d_300            7天>$300次数
# 54 is_spike_500            [CAT] 是否>$500
# 55 spike_count_24h_500           24小时>$500次数
# 56 spike_count_7d_500            7天>$500次数
# 57 price_vs_max_24h              当前/24小时最高
# 58 price_vs_min_24h              当前/24小时最低
# 59 price_range_24h               24小时价格范围
# 60 price_percentile_24h          24小时百分位

# --- 波动率 (4个) ---
# 61 cv_24h                        24小时变异系数
# 62 amplitude_1h                  1小时振幅
# 63 volatility_change             波动率变化
# 64 recent_volatility_ratio       近期波动率比

# --- Horizon (3个) ---
# 65 target_hour             [CAT] 目标时刻小时
# 66 target_is_peak          [CAT] 目标是否峰时
# 67 crosses_hour                  是否跨小时

# =============================================================================
# DAM特征 (12个)
# =============================================================================

# --- DAM当前特征 (6个) ---
# 68 dam_price_current             当前小时DAM价格
# 69 rtm_dam_spread                RTM-DAM价差
# 70 spread_mean_1h                1小时平均价差
# 71 spread_mean_24h               24小时平均价差
# 72 spread_std_24h                24小时价差标准差
# 73 rtm_dam_ratio                 RTM/DAM比率

# --- DAM目标特征 (6个) ---
# 74 dam_price_target              目标时刻DAM价格
# 75 dam_change                    DAM变化
# 76 dam_change_pct                DAM变化率
# 77 dam_target_zscore             目标DAM的Z-score
# 78 dam_target_above_100    [CAT] 目标DAM是否>$100
# 79 dam_target_above_200    [CAT] 目标DAM是否>$200


import pandas as pd
import numpy as np
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

try:
    import holidays
    US_HOLIDAYS = holidays.US(years=range(2015, 2026))
except ImportError:
    US_HOLIDAYS = None


# =============================================================================
# 配置类
# =============================================================================

@dataclass
class FeatureConfig:
    """特征工程配置"""
    horizons: List[int] = field(default_factory=lambda: [1, 2, 3, 4])
    target_type: str = 'delta'  # 'absolute', 'delta'
    spike_thresholds: List[int] = field(default_factory=lambda: [100, 300, 500])
    peak_hours: List[int] = field(default_factory=lambda: list(range(7, 23)))
    summer_months: List[int] = field(default_factory=lambda: [6, 7, 8, 9])
    super_peak_hours: List[int] = field(default_factory=lambda: list(range(14, 21)))
    morning_ramp_hours: List[int] = field(default_factory=lambda: list(range(5, 10)))
    evening_ramp_hours: List[int] = field(default_factory=lambda: list(range(17, 21)))


# =============================================================================
# 分类特征定义
# =============================================================================

# RTM分类特征 (21个)
RTM_CATEGORICAL_FEATURES = [
    'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'quarter',
    'is_weekend', 'is_peak_hour', 'is_summer', 'is_super_peak',
    'is_holiday', 'is_business_day', 'hour_block',
    'hour_dow_interaction', 'is_morning_ramp', 'is_evening_ramp',
    'is_spike_100', 'is_spike_300', 'is_spike_500',
    'target_hour', 'target_is_peak',
]

# DAM分类特征 (2个)
DAM_CATEGORICAL_FEATURES = [
    'dam_target_above_100',
    'dam_target_above_200',
]

# 全部分类特征
CATEGORICAL_FEATURES = RTM_CATEGORICAL_FEATURES + DAM_CATEGORICAL_FEATURES


# =============================================================================
# 主类
# =============================================================================

class ERCOTFeatureEngineer:
    """ERCOT RTM+DAM 完整特征工程器"""
    
    def __init__(self, config: FeatureConfig = None):
        self.config = config or FeatureConfig()
        self.feature_names: Dict[int, List[str]] = {}
        self.dam_map: Optional[Dict] = None
        
    # =========================================================================
    # 数据预处理
    # =========================================================================
    
    def preprocess_rtm(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        预处理RTM原始数据
        
        Parameters
        ----------
        df : pd.DataFrame
            原始RTM数据，4列: date, hour(1-24), interval(1-4), price
        """
        df = df.copy()
        df.columns = ['date', 'hour_raw', 'interval', 'price']
        
        df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
        df['hour'] = df['hour_raw'] - 1
        df['minute'] = (df['interval'] - 1) * 15
        
        df['timestamp'] = (df['date'] + 
                          pd.to_timedelta(df['hour'], unit='h') + 
                          pd.to_timedelta(df['minute'], unit='m'))
        
        result = df.set_index('timestamp').sort_index()[['price']]
        
        print(f"RTM数据预处理完成:")
        print(f"  时间范围: {result.index.min()} ~ {result.index.max()}")
        print(f"  数据量: {len(result):,} 条 (15分钟频率)")
        print(f"  价格范围: ${result['price'].min():.2f} ~ ${result['price'].max():.2f}")
        
        return result
    
    def preprocess_dam(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        预处理DAM原始数据
        
        Parameters
        ----------
        df : pd.DataFrame
            原始DAM数据，3列: date, hour(1-24), dam_price
        """
        df = df.copy()
        df.columns = ['date', 'hour_raw', 'dam_price']
        
        df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
        df['hour'] = df['hour_raw'] - 1  # HE1 -> hour 0
        
        df['timestamp'] = df['date'] + pd.to_timedelta(df['hour'], unit='h')
        
        result = df.set_index('timestamp').sort_index()[['dam_price']]
        
        # 创建DAM价格映射 (用于快速查找)
        self.dam_map = result['dam_price'].to_dict()
        
        print(f"DAM数据预处理完成:")
        print(f"  时间范围: {result.index.min()} ~ {result.index.max()}")
        print(f"  数据量: {len(result):,} 条 (1小时频率)")
        print(f"  价格范围: ${result['dam_price'].min():.2f} ~ ${result['dam_price'].max():.2f}")
        
        return result
    
    # =========================================================================
    # RTM特征构建 (68个)
    # =========================================================================
    
    def _add_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加时间特征 (17个)"""
        idx = df.index
        cfg = self.config
        
        # 基础时间 (7个)
        df['hour'] = idx.hour
        df['minute'] = idx.minute
        df['day_of_week'] = idx.dayofweek
        df['day_of_month'] = idx.day
        df['month'] = idx.month
        df['quarter'] = idx.quarter
        df['week_of_year'] = idx.isocalendar().week.astype(int)
        
        # 时间标记 (7个)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_peak_hour'] = df['hour'].isin(cfg.peak_hours).astype(int)
        df['is_summer'] = df['month'].isin(cfg.summer_months).astype(int)
        df['is_super_peak'] = (
            df['is_summer'].astype(bool) & 
            df['hour'].isin(cfg.super_peak_hours) & 
            ~df['is_weekend'].astype(bool)
        ).astype(int)
        
        if US_HOLIDAYS is not None:
            holiday_dates = set(US_HOLIDAYS.keys())
            df['is_holiday'] = idx.date
            df['is_holiday'] = df['is_holiday'].apply(lambda x: 1 if x in holiday_dates else 0)
        else:
            df['is_holiday'] = self._simple_holiday_flag(idx)
        
        df['is_business_day'] = (
            ~df['is_weekend'].astype(bool) & 
            ~df['is_holiday'].astype(bool)
        ).astype(int)
        
        df['hour_block'] = pd.cut(
            df['hour'], bins=[-1, 6, 12, 18, 24], labels=[0, 1, 2, 3]
        ).astype(int)
        
        # 时间交互 (3个)
        df['hour_dow_interaction'] = df['hour'] * 10 + df['day_of_week']
        
        df['is_morning_ramp'] = (
            df['hour'].isin(cfg.morning_ramp_hours) & 
            df['is_business_day'].astype(bool)
        ).astype(int)
        
        df['is_evening_ramp'] = (
            df['hour'].isin(cfg.evening_ramp_hours) & 
            df['is_summer'].astype(bool)
        ).astype(int)
        
        return df
    
    def _simple_holiday_flag(self, idx: pd.DatetimeIndex) -> pd.Series:
        """简化版假日标记"""
        holidays_list = []
        for year in range(2015, 2026):
            holidays_list.extend([
                f'{year}-01-01', f'{year}-07-04', f'{year}-12-25',
                f'{year}-12-31', f'{year}-11-25', f'{year}-11-26',
            ])
        holidays_set = set(pd.to_datetime(holidays_list).date)
        return pd.Series(idx.date, index=idx).isin(holidays_set).astype(int)
    
    def _add_lag_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加滞后特征 (8个)"""
        price = df['price']
        
        # 基础滞后 (6个)
        df['price_lag_1'] = price.shift(1)
        df['price_lag_2'] = price.shift(2)
        df['price_lag_3'] = price.shift(3)
        df['price_lag_4'] = price.shift(4)
        df['price_lag_96'] = price.shift(96)
        df['price_lag_672'] = price.shift(672)
        
        # 滞后交互 (2个)
        df['price_lag_4_diff_lag_96'] = df['price_lag_4'] - df['price_lag_96']
        df['price_lag_672_diff_lag_96'] = df['price_lag_672'] - df['price_lag_96']
        
        return df
    
    def _add_rolling_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加滚动统计特征 (13个)"""
        price = df['price']
        
        # 1小时滚动 (4个)
        roll_1h = price.rolling(4, min_periods=1)
        df['price_mean_1h'] = roll_1h.mean()
        df['price_std_1h'] = roll_1h.std()
        df['price_max_1h'] = roll_1h.max()
        df['price_min_1h'] = roll_1h.min()
        
        # 3小时滚动 (2个)
        roll_3h = price.rolling(12, min_periods=4)
        df['price_mean_3h'] = roll_3h.mean()
        df['price_std_3h'] = roll_3h.std()
        
        # 24小时滚动 (5个)
        roll_24h = price.rolling(96, min_periods=24)
        df['price_mean_24h'] = roll_24h.mean()
        df['price_std_24h'] = roll_24h.std()
        df['price_max_24h'] = roll_24h.max()
        df['price_min_24h'] = roll_24h.min()
        df['price_skew_24h'] = roll_24h.skew()
        
        # 7天滚动 (2个)
        roll_7d = price.rolling(672, min_periods=168)
        df['price_mean_7d'] = roll_7d.mean()
        df['price_std_7d'] = roll_7d.std()
        
        return df
    
    def _add_same_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加同时刻历史统计特征 (2个)"""
        price = df['price']
        
        lags_7d = [price.shift(96 * i) for i in range(1, 8)]
        df['same_time_mean'] = pd.concat(lags_7d, axis=1).mean(axis=1)
        
        lags_4w = [price.shift(672 * i) for i in range(1, 5)]
        df['same_time_dow_mean'] = pd.concat(lags_4w, axis=1).mean(axis=1)
        
        return df
    
    def _add_momentum_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加动量特征 (8个)"""
        price = df['price']
        
        # 差分 (3个)
        df['price_diff_1'] = price.diff(1)
        df['price_diff_4'] = price.diff(4)
        df['price_diff_96'] = price.diff(96)
        
        # 变化率 (1个)
        df['price_pct_1'] = price.pct_change(1)
        
        # 动量 (2个)
        mean_1h = price.rolling(4, min_periods=1).mean()
        std_1h = price.rolling(4, min_periods=1).std()
        df['price_momentum_1h'] = (price - mean_1h) / (std_1h + 1e-6)
        
        mean_24h = price.rolling(96, min_periods=24).mean()
        std_24h = price.rolling(96, min_periods=24).std()
        df['price_momentum_24h'] = (price - mean_24h) / (std_24h + 1e-6)
        
        # 相对位置 (1个)
        df['price_vs_mean_24h'] = price / (mean_24h + 1e-6)
        
        # 加速度 (1个)
        df['price_acceleration'] = df['price_diff_1'].diff(1)
        
        return df
    
    def _add_spike_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加尖峰特征 (13个)"""
        price = df['price']
        
        # 尖峰标记和计数 (9个)
        for threshold in self.config.spike_thresholds:
            col = f'is_spike_{threshold}'
            df[col] = (price > threshold).astype(int)
            df[f'spike_count_24h_{threshold}'] = df[col].rolling(96, min_periods=1).sum()
            df[f'spike_count_7d_{threshold}'] = df[col].rolling(672, min_periods=1).sum()
        
        # 极值特征 (4个)
        max_24h = price.rolling(96, min_periods=24).max()
        min_24h = price.rolling(96, min_periods=24).min()
        
        df['price_vs_max_24h'] = price / (max_24h + 1e-6)
        df['price_vs_min_24h'] = price / (min_24h + 1e-6)
        df['price_range_24h'] = max_24h - min_24h
        df['price_percentile_24h'] = price.rolling(96, min_periods=24).apply(
            lambda x: (x[-1] > x[:-1]).sum() / (len(x) - 1) if len(x) > 1 else 0.5, raw=True
        )
        
        return df
    
    def _add_volatility_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """添加波动率特征 (4个)"""
        price = df['price']
        
        mean_24h = price.rolling(96, min_periods=24).mean()
        std_24h = price.rolling(96, min_periods=24).std()
        std_1h = price.rolling(4, min_periods=1).std()
        
        df['cv_24h'] = std_24h / (mean_24h + 1e-6)
        df['amplitude_1h'] = price.rolling(4, min_periods=1).max() - price.rolling(4, min_periods=1).min()
        df['volatility_change'] = std_1h.diff(4)
        df['recent_volatility_ratio'] = std_1h / (std_24h + 1e-6)
        
        return df
    
    def _add_horizon_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """添加Horizon特征 (3个)"""
        target_time = df.index + pd.Timedelta(minutes=horizon * 15)
        
        df['target_hour'] = target_time.hour
        df['target_is_peak'] = pd.Series(target_time.hour, index=df.index).isin(
            self.config.peak_hours
        ).astype(int)
        df['crosses_hour'] = (df.index.hour != target_time.hour).astype(int)
        
        return df
    
    # =========================================================================
    # DAM特征构建 (12个)
    # =========================================================================
    
    def _add_dam_features(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """
        添加DAM相关特征 (12个)
        
        关键点: 预测未来RTM时，对应时刻的DAM价格是已知的！
        """
        if self.dam_map is None:
            print("警告: DAM数据未加载，跳过DAM特征")
            return df
        
        # -----------------------------------------
        # 1. 当前时刻的DAM特征
        # -----------------------------------------
        # 当前RTM时刻对应的小时
        hour_start = df.index.floor('h')
        df['dam_price_current'] = pd.Series(hour_start, index=df.index).map(self.dam_map)
        
        # 当前RTM vs DAM价差
        df['rtm_dam_spread'] = df['price'] - df['dam_price_current']
        
        # 价差统计
        df['spread_mean_1h'] = df['rtm_dam_spread'].rolling(4, min_periods=1).mean()
        df['spread_mean_24h'] = df['rtm_dam_spread'].rolling(96, min_periods=24).mean()
        df['spread_std_24h'] = df['rtm_dam_spread'].rolling(96, min_periods=24).std()
        
        # RTM/DAM比率
        df['rtm_dam_ratio'] = df['price'] / (df['dam_price_current'] + 1)
        
        # -----------------------------------------
        # 2. 目标时刻的DAM特征 (关键！这是已知的未来信息)
        # -----------------------------------------
        target_time = df.index + pd.Timedelta(minutes=horizon * 15)
        target_hour = target_time.floor('h')
        df['dam_price_target'] = pd.Series(target_hour, index=df.index).map(self.dam_map)
        
        # -----------------------------------------
        # 3. DAM衍生特征
        # -----------------------------------------
        # DAM变化 (目标时刻 vs 当前)
        df['dam_change'] = df['dam_price_target'] - df['dam_price_current']
        df['dam_change_pct'] = df['dam_change'] / (df['dam_price_current'] + 1)
        
        # 目标DAM的相对位置
        dam_mean = df['dam_price_current'].rolling(96, min_periods=24).mean()
        dam_std = df['dam_price_current'].rolling(96, min_periods=24).std()
        df['dam_target_zscore'] = (df['dam_price_target'] - dam_mean) / (dam_std + 1)
        
        # 目标DAM高价标记
        df['dam_target_above_100'] = (df['dam_price_target'] > 100).astype(int)
        df['dam_target_above_200'] = (df['dam_price_target'] > 200).astype(int)
        
        return df
    
    # =========================================================================
    # 目标变量构建
    # =========================================================================
    
    def _add_target(self, df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        """构建目标变量"""
        price = df['price']
        future_price = price.shift(-horizon)
        
        if self.config.target_type == 'absolute':
            df['target'] = future_price
        elif self.config.target_type == 'delta':
            df['target'] = future_price - price
        
        df['actual_future_price'] = future_price
        df['current_price'] = price
        
        return df
    
    # =========================================================================
    # 主入口
    # =========================================================================
    
    def build_features(
        self, 
        rtm_df: pd.DataFrame, 
        dam_df: Optional[pd.DataFrame],
        horizon: int
    ) -> pd.DataFrame:
        """
        构建完整特征集 (RTM + DAM)
        
        Parameters
        ----------
        rtm_df : pd.DataFrame
            RTM价格数据 (已预处理)
        dam_df : pd.DataFrame or None
            DAM价格数据 (已预处理)，如果为None则只构建RTM特征
        horizon : int
            预测步数 (1-4)
        """
        # 预处理DAM数据
        if dam_df is not None and self.dam_map is None:
            self.dam_map = dam_df['dam_price'].to_dict()
        
        result = rtm_df.copy()
        
        # RTM特征 (68个)
        result = self._add_time_features(result)
        result = self._add_lag_features(result)
        result = self._add_rolling_features(result)
        result = self._add_same_time_features(result)
        result = self._add_momentum_features(result)
        result = self._add_spike_features(result)
        result = self._add_volatility_features(result)
        result = self._add_horizon_features(result, horizon)
        
        # DAM特征 (12个)
        if self.dam_map is not None:
            result = self._add_dam_features(result, horizon)
        
        # 目标变量
        result = self._add_target(result, horizon)
        
        # 记录特征名
        exclude = {'price', 'target', 'actual_future_price', 'current_price'}
        self.feature_names[horizon] = [c for c in result.columns if c not in exclude]
        
        return result
    
    def build_all_horizons(
        self, 
        rtm_df: pd.DataFrame, 
        dam_df: Optional[pd.DataFrame] = None
    ) -> Dict[int, pd.DataFrame]:
        """为所有horizon构建特征集"""
        results = {}
        for horizon in self.config.horizons:
            print(f"\n构建 Horizon={horizon} ({horizon*15}分钟) 特征...")
            results[horizon] = self.build_features(rtm_df, dam_df, horizon)
            print(f"  特征数: {len(self.feature_names[horizon])}")
        return results
    
    def get_feature_info(self, horizon: int) -> None:
        """打印特征信息"""
        if horizon not in self.feature_names:
            print(f"Horizon={horizon} 的特征尚未构建")
            return
        
        features = self.feature_names[horizon]
        cat_features = [f for f in features if f in CATEGORICAL_FEATURES]
        
        # 区分RTM和DAM特征
        dam_feature_names = [
            'dam_price_current', 'dam_price_target', 'rtm_dam_spread',
            'spread_mean_1h', 'spread_mean_24h', 'spread_std_24h',
            'rtm_dam_ratio', 'dam_change', 'dam_change_pct',
            'dam_target_zscore', 'dam_target_above_100', 'dam_target_above_200'
        ]
        dam_features = [f for f in features if f in dam_feature_names]
        rtm_features = [f for f in features if f not in dam_feature_names]
        
        print(f"\n{'='*60}")
        print(f"Horizon={horizon} ({horizon*15}分钟) 特征统计")
        print(f"{'='*60}")
        print(f"  RTM特征: {len(rtm_features)}")
        print(f"  DAM特征: {len(dam_features)}")
        print(f"  总特征数: {len(features)}")
        print(f"  分类特征: {len(cat_features)}")
        print(f"  数值特征: {len(features) - len(cat_features)}")
    
    def get_categorical_indices(self, horizon: int) -> List[int]:
        """获取分类特征索引"""
        if horizon not in self.feature_names:
            return []
        return [i for i, f in enumerate(self.feature_names[horizon]) if f in CATEGORICAL_FEATURES]


# =============================================================================
# 数据划分
# =============================================================================

def split_data(
    df: pd.DataFrame,
    train_end: str = '2022-12-31',
    val_end: str = '2023-12-31',
    feature_cols: List[str] = None
) -> Dict:
    """按时间划分数据集"""
    required = ['target'] + (feature_cols or [])
    df_clean = df.dropna(subset=[c for c in required if c in df.columns])
    
    train = df_clean[df_clean.index <= train_end]
    val = df_clean[(df_clean.index > train_end) & (df_clean.index <= val_end)]
    test = df_clean[df_clean.index > val_end]
    
    print(f"\n数据划分:")
    print(f"  训练集: {len(train):>8,} 条 ({train.index.min().date()} ~ {train.index.max().date()})")
    print(f"  验证集: {len(val):>8,} 条 ({val.index.min().date()} ~ {val.index.max().date()})")
    print(f"  测试集: {len(test):>8,} 条 ({test.index.min().date()} ~ {test.index.max().date()})")
    
    return {'train': train, 'val': val, 'test': test, 'feature_cols': feature_cols}

def load_data(filepath: str) -> pd.DataFrame:
    """
    加载原始数据
    
    Parameters
    ----------
    filepath : str
        CSV文件路径
    
    Returns
    -------
    pd.DataFrame
        原始数据
    """
    df = pd.read_csv(filepath)
    df.columns = ['date', 'hour', 'interval', 'price']
    return df

/home/admin01/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [15]:
if __name__ == "__main__":
    
    # 获取RTM数据源（15分钟频率）
    rtm_raw = load_data('./processed_data/LZ_HOUSTON_price_2025.csv')
    rtm_raw = rtm_raw.iloc[:-96*10]
    print(f"RTM原始数据: {len(rtm_raw):,} 条")
    
    # 或群DAM数据源 (1小时频率)
    dam_raw = pd.read_csv("../DAM_Forecast/processed_data/dam_HOUSTON_price.csv")
    print(f"DAM原始数据: {len(dam_raw):,} 条")
    
    # =========================================================================
    # 特征工程
    # =========================================================================
    print("\n" + "="*70)
    print("开始特征工程")
    print("="*70)
    
    config = FeatureConfig(target_type='delta')
    fe = ERCOTFeatureEngineer(config)
    
    # 预处理
    rtm_df = fe.preprocess_rtm(rtm_raw)
    dam_df = fe.preprocess_dam(dam_raw)
    
    # 构建所有horizon的特征
    all_features = fe.build_all_horizons(rtm_df, dam_df)
    
    # 打印特征信息
    for horizon in [1, 2, 3, 4]:
        fe.get_feature_info(horizon)
    
    # =========================================================================
    # 数据划分
    # =========================================================================
    print("\n" + "="*70)
    print("数据划分 (Horizon=1)")
    print("="*70)
    
    df_h1 = all_features[1]
    feature_cols = fe.feature_names[1]
    data_splits = split_data(df_h1, feature_cols=feature_cols)
    
    # =========================================================================
    # 输出特征列表
    # =========================================================================
    print("\n" + "="*70)
    print("完整特征列表")
    print("="*70)
    
    cat_indices = fe.get_categorical_indices(1)
    print(f"\n分类特征索引 ({len(cat_indices)}个): {cat_indices}")
    
    # 按分组打印
    groups = {
        'RTM-时间特征 (7)': ['hour', 'minute', 'day_of_week', 'day_of_month', 'month', 
                           'quarter', 'week_of_year'],
        'RTM-时间标记 (7)': ['is_weekend', 'is_peak_hour', 'is_summer', 'is_super_peak',
                           'is_holiday', 'is_business_day', 'hour_block'],
        'RTM-时间交互 (3)': ['hour_dow_interaction', 'is_morning_ramp', 'is_evening_ramp'],
        'RTM-滞后特征 (6)': ['price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_4',
                           'price_lag_96', 'price_lag_672'],
        'RTM-滞后交互 (2)': ['price_lag_4_diff_lag_96', 'price_lag_672_diff_lag_96'],
        'RTM-滚动统计 (13)': ['price_mean_1h', 'price_std_1h', 'price_max_1h', 'price_min_1h',
                            'price_mean_3h', 'price_std_3h', 'price_mean_24h', 'price_std_24h',
                            'price_max_24h', 'price_min_24h', 'price_skew_24h',
                            'price_mean_7d', 'price_std_7d'],
        'RTM-同时刻统计 (2)': ['same_time_mean', 'same_time_dow_mean'],
        'RTM-动量特征 (8)': ['price_diff_1', 'price_diff_4', 'price_diff_96', 'price_pct_1',
                           'price_momentum_1h', 'price_momentum_24h', 'price_vs_mean_24h',
                           'price_acceleration'],
        'RTM-尖峰特征 (13)': ['is_spike_100', 'spike_count_24h_100', 'spike_count_7d_100',
                            'is_spike_300', 'spike_count_24h_300', 'spike_count_7d_300',
                            'is_spike_500', 'spike_count_24h_500', 'spike_count_7d_500',
                            'price_vs_max_24h', 'price_vs_min_24h', 'price_range_24h',
                            'price_percentile_24h'],
        'RTM-波动率 (4)': ['cv_24h', 'amplitude_1h', 'volatility_change', 'recent_volatility_ratio'],
        'RTM-Horizon (3)': ['target_hour', 'target_is_peak', 'crosses_hour'],
        'DAM-当前特征 (6)': ['dam_price_current', 'rtm_dam_spread', 'spread_mean_1h',
                           'spread_mean_24h', 'spread_std_24h', 'rtm_dam_ratio'],
        'DAM-目标特征 (6)': ['dam_price_target', 'dam_change', 'dam_change_pct',
                           'dam_target_zscore', 'dam_target_above_100', 'dam_target_above_200'],
    }
    
    print(f"\n完整特征列表 ({len(feature_cols)}个):")
    print("-" * 60)
    
    total = 0
    for group_name, group_features in groups.items():
        existing = [f for f in group_features if f in feature_cols]
        if existing:
            print(f"\n{group_name}:")
            for f in existing:
                cat_mark = "[CAT]" if f in CATEGORICAL_FEATURES else ""
                idx = feature_cols.index(f)
                print(f"  {idx:2d}. {f:<30} {cat_mark}")
            total += len(existing)

    print(f"\n{'-'*60}")
    print(f"总计: {total} 个特征 (RTM: 68, DAM: 12)")
    
    # =========================================================================
    # 验证DAM特征
    # =========================================================================
    print("\n" + "="*70)
    print("DAM特征验证")
    print("="*70)
    
    sample = df_h1.dropna().iloc[1000:1005]
    print("\n样本数据:")
    dam_cols = ['price', 'dam_price_current', 'dam_price_target', 'rtm_dam_spread', 'dam_change']
    print(sample[dam_cols].round(2).to_string())
    
    print("\n验证说明:")
    print("  - dam_price_current: 当前小时的DAM价格")
    print("  - dam_price_target: 目标时刻的DAM价格 (horizon=1时为下一小时)")
    print("  - rtm_dam_spread: price - dam_price_current")
    print("  - dam_change: dam_price_target - dam_price_current")

RTM原始数据: 524,064 条
DAM原始数据: 95,928 条

开始特征工程
RTM数据预处理完成:
  时间范围: 2011-01-01 00:00:00 ~ 2025-12-10 23:45:00
  数据量: 524,064 条 (15分钟频率)
  价格范围: $-61.85 ~ $9314.49
DAM数据预处理完成:
  时间范围: 2015-01-01 00:00:00 ~ 2025-12-10 23:00:00
  数据量: 95,928 条 (1小时频率)
  价格范围: $-3.83 ~ $9011.25

构建 Horizon=1 (15分钟) 特征...
  特征数: 80

构建 Horizon=2 (30分钟) 特征...
  特征数: 80

构建 Horizon=3 (45分钟) 特征...
  特征数: 80

构建 Horizon=4 (60分钟) 特征...
  特征数: 80

Horizon=1 (15分钟) 特征统计
  RTM特征: 68
  DAM特征: 12
  总特征数: 80
  分类特征: 23
  数值特征: 57

Horizon=2 (30分钟) 特征统计
  RTM特征: 68
  DAM特征: 12
  总特征数: 80
  分类特征: 23
  数值特征: 57

Horizon=3 (45分钟) 特征统计
  RTM特征: 68
  DAM特征: 12
  总特征数: 80
  分类特征: 23
  数值特征: 57

Horizon=4 (60分钟) 特征统计
  RTM特征: 68
  DAM特征: 12
  总特征数: 80
  分类特征: 23
  数值特征: 57

数据划分 (Horizon=1)

数据划分:
  训练集:  280,234 条 (2015-01-01 ~ 2022-12-31)
  验证集:   34,956 条 (2022-12-31 ~ 2023-12-31)
  测试集:   68,041 条 (2023-12-31 ~ 2025-12-10)

完整特征列表

分类特征索引 (23个): [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66, 78, 

In [16]:
# 特征数据保存
fea1 = all_features[1].iloc[:, 1:].dropna()
fea1['date'] = fea1.index
fea1.to_csv('./rtm_dam_fea/train_data_houston_horizon1.csv', index=None)

fea2 = all_features[2].iloc[:, 1:].dropna()
fea2['date'] = fea2.index
fea2.to_csv('./rtm_dam_fea/train_data_houston_horizon2.csv', index=None)

fea3 = all_features[3].iloc[:, 1:].dropna()
fea3['date'] = fea3.index
fea3.to_csv('./rtm_dam_fea/train_data_houston_horizon3.csv', index=None)

fea4 = all_features[4].iloc[:, 1:].dropna()
fea4['date'] = fea4.index
fea4.to_csv('./rtm_dam_fea/train_data_houston_horizon4.csv', index=None)